## CK

### Assignment

Create a chat like the one in this week’s [RAG with HuggingFace](https://medium.com/@imatasneemkoushar/developing-rag-app-in-google-colab-with-hugging-face-a9dd0dd7e05f) reading. Instead of using a PDF file as the input, using any Wikipedia page. You can read the page using many techniques, including Beautiful Soup. In the end, the input is text no matter where you get it from.

In [ ]:
!pip install beautifulsoup4 requests sentence-transformers faiss-cpu transformers accelerate

In [2]:
import requests
import os
from google.colab import userdata
from bs4 import BeautifulSoup
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

First I will begin by creating a function, using BeautifulSoup, to scrape a Wikipedia page.

In [3]:
def scrape_wikipedia(url):
    """fetch text content from any Wikipedia page by extracting all body paragraphs."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Failed to load page. Status code: {response.status_code}")

    soup = BeautifulSoup(response.content, 'html.parser')

    # remove unwanted elements like reference markers, tables, and edit links
    for element in soup.find_all(['sup', 'table', 'style', 'script']):
        element.decompose()

    # find all main content paragraphs
    paragraphs = soup.find_all('p')

    docs = []
    section_counter = 1
    for p in paragraphs:
        # clean text and remove whitespace
        text = re.sub(r'\s+', ' ', p.get_text()).strip()

        # save any paragraphs that are not empty
        if len(text) > 40:
            docs.append({"section_id": section_counter, "text": text})
            section_counter += 1

    return docs

Now I need to create the function that is able to chunk the text from the wikipedia text that was scraped. Here, I'll pass in the URL of the page I want to scrape. I will use the Wikipedia page of my favorite author, [Jane Austen](https://en.wikipedia.org/wiki/Jane_Austen$0).

In [4]:
# simple chunking function from the article
def chunk_text(text, max_chars=500, overlap=100):
    """split long text into smaller chunks (500 char) with overlap (100 char) chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        start += max_chars - overlap
    return chunks

# the url I want to use
wiki_url = "https://en.wikipedia.org/wiki/Jane_Austen"
wiki_docs = scrape_wikipedia(wiki_url)
print(f"Scraped {len(wiki_docs)} paragraphs from Wikipedia.")

# create text chunks - modified from article
chunks = []
for doc in wiki_docs:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "section_id": doc["section_id"],
            "text": chunk
        })

print(f"Total chunks created: {len(chunks)}")
print("Sample chunk:", chunks[0])

Scraped 83 paragraphs from Wikipedia.
Total chunks created: 179
Sample chunk: {'section_id': 1, 'text': 'Jane Austen (/ˈɒstɪn, ˈɔːstɪn/ OST-in, AW-stin; 16 December 1775 – 18 July 1817) was an English writer known primarily for her six novels, which implicitly interpret, critique, and comment on the English landed gentry at the end of the 18th century.'}


Now, we need to use an embedder, the sentence transformer, all-MiniLM-L6-v2, and then generate the vector embeddings.

FAISS is a tool designed for similarity searches for vectors. It came from FAIR (now Meta) research labs. It is not a database, or a service, or anything like that. Matthijs Douze and colleagues describe it as "a collection of highly optimized algorithms and indexing methods focused squarely on the algorithmic challenges of Approximate Nearest Neighbor Search (ANNS). (Leo, 2025). FAISS is used to search vectors and efficiently retrieve answers for RAG. It does semantic searches and not keyword searches, so it should be to find words or ideas that are *close* to what is queried by the user.

FAISS was built to make vector searches fast and scalable. Exact nearest neighbor searches can be slow and require a good deal of computational power (especially if you have a large dataset), which is why the Approximate Nearest Neighbor Search (ANNS) is used. It doesn't guarantee the absolute nearest neighbor, but instead will try to find a vector that is close to the nearest neighbor, but do is very quickly. This means we sacrifice a little bit of perfect accuracy for a "massive gain speed." FAISS supports a few distance metrics but commonly used for context embeddings, and therefore RAG, are Euclidean distance and cosine similarity. Euclidean distance is the default in FAISS for comparing vectors.

For indexing, specifically for RAG, two of the most used techniques are Inverted File (IVF) and Hierarchical Navigable Small World (HNSW). After indexing, FAISS then utilizes a number of vector compression techniques to reduce the required memory of the index and helps to enable faster computations for distance.

In [5]:
# following along with the article, setting up the embeddings + vector index
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# generate vector embeddings
embeddings = embedder.encode(
    [c["text"] for c in chunks],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# build FAISS flat (exact)
# IP=Inner Product; uses dot product math to calculate cosine similarity between vectors
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS Index size:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index size: 179


Now there needs to be a retrieval function that will find the top n number of text chunks based on a query.



In [ ]:
# retrieval function from article
def retrieve(query, topk=5):
    """retrieve top-k relevant text chunks for a given query."""
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    scores, idxs = index.search(q_emb, topk)

    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({**chunks[idx], "score": float(score)})
    return results

Now use a Hugging Face LLM to do the Q&A. I had an error with the text2text-generation (KeyError) so a better option is to use the `AutoModelForSeq2SeqLM` which is the recommended option for the flan-tf-large tokenizer. It also provides more control over the `generate()` parameters for encoder-decoder architectures like Flan-T5.

In [ ]:
# taking a slight detour from the article, text2text-generation returned a KeyError
# swapping to AutoModelForSeq2SeqLM as the recommended model

model_name = "google/flan-t5-large"

# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto"
)

# answer function from article, slightly modified for the wikipedia content
def answer(query, topk=5):
    """retrieves context and passes it along with the query to the LLM."""
    ctx = retrieve(query, topk)
    context_text = "\n\n".join([f"[Section {c['section_id']}] {c['text']}" for c in ctx])

    prompt = f"""Answer the question based only on the context below.
If you don't know, say you don't know.

Question: {query}

Context:
{context_text}"""

    # tokenize prompt and map to input tokens, PyTorch Tensor format
    # then move input tensor to same device model is on
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # generate response; greedy search
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    # decode output tokens back to text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated_text, ctx

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


I did try setting `tie_word_embeddings=False` and still had an error. the `google/flan-t5-large` stores its token embedding weights under a single shared tensor called `shared.weight` instead of separate keys of `encoder.embed_tokens.weight` and `decoder.embed_tokens.weight`. This means that when I pass the `tie_word_embeddings=False` the loader also expects the additional encoder and decoder keys. To fix this either explicitly set `tie_word_embeddings=True` or leave it out, since the default value is true. This let's the framework tie the encoder and decoder embed_tokens back to the model's `shared.weight` matrix.

Now setting up for the chat

In [ ]:
print("\n*** Jane Austen Wikipedia RAG Chatbot ***")
# loop to continue prompting user for question, provide answer + citations
# based on article
while True:
    q = input("\nAsk a question about Jane Austen (or type 'quit' to exit): ")
    if q.lower() == "quit":
        break

    resp, ctx = answer(q)
    print("\nAnswer:", resp)
    print("Sources (Paragraph Sections):", [f"Section {c['section_id']}" for c in ctx])
    print("=" * 50)


*** Jane Austen Wikipedia RAG Chatbot ***

Ask a question about Jane Austen (or type 'quit' to exit): What year was Jane Austen born


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (586 > 512). Running this sequence through the model will result in indexing errors



Answer: 1775
Sources (Paragraph Sections): ['Section 10', 'Section 1', 'Section 72', 'Section 4', 'Section 26']

Ask a question about Jane Austen (or type 'quit' to exit): When did she die

Answer: May
Sources (Paragraph Sections): ['Section 47', 'Section 54', 'Section 52', 'Section 10', 'Section 52']

Ask a question about Jane Austen (or type 'quit' to exit): how many novels did she write

Answer: six novels
Sources (Paragraph Sections): ['Section 47', 'Section 44', 'Section 57', 'Section 23', 'Section 3']

Ask a question about Jane Austen (or type 'quit' to exit): what did he plots often explore

Answer: an adult woman whose intelligence and force of character are greater than those of anyone she encounters
Sources (Paragraph Sections): ['Section 46', 'Section 68', 'Section 60', 'Section 68', 'Section 28']

Ask a question about Jane Austen (or type 'quit' to exit): how old was she when she was published

Answer: eleven
Sources (Paragraph Sections): ['Section 4', 'Section 23', 'Secti

==========================================================================================================================================================================

This was a fun exercise, and clearly I enjoyed spending time validating the results.

#### Going through each of the questions I asked and the results.

**Q** What year was Jane Austen born?

**A** 1775

**Analysis** This was correct, she was born 16 December 1775

==========================================================================================================================================================================

**Q** When did she die

**A** May

**Analysis** Not even close, she died 18 July 1817

==========================================================================================================================================================================

**Q** how many novels did she write

**A** six novels

**Analysis** This is correct.

==========================================================================================================================================================================

**Q** what did he plots often explore

**A** an adult woman whose intelligence and force of character are greater than those of anyone she encounters

**Analysis** This is really off. Direct quote from Wikipedia, "Austen's plots often explore the dependence of women on marriage for the pursuit of favourable social standing and economic security." No where is intelligence or force of character mentioned about her typical plot lines.

==========================================================================================================================================================================

**Q** how old was she when she was published

**A** eleven

**Analysis** Incorrect, she began writing poems and stories for her family and to "amuse herself". She wrote novels before she was 22 but she was not published until she was 35. Not really sure how the model came up with eleven as the answer, since only one paragraph mentions her being eleven, and that is only to say how she wrote poems and stories for herself. The sentence I had expected the model to find was, "Austen wrote major novels before the age of 22, but she was not published until she was 35."

==========================================================================================================================================================================

**Q** how many letters did she write that survive

**A** 160

**Analysis** Correct, they believe she wrote approximately 3,000 letters over her lifetime but only about 160 remain.

==========================================================================================================================================================================

**Q** what was her sister's name

**A** Cassandra

**Analysis** Correct.

==========================================================================================================================================================================

**Q** who wrote the first biography on her

**A** George Pellew

**Analysis** Incorrect. The first biography on her was written by her brother Henry Thomas Austen, in 1818 which was a "Biographical Notice" and appeared in a posthumous edition of Northanger Abbey. It included extracts from two of her letters. Some family members did not want it to be published.

==========================================================================================================================================================================

**Q** what was her father's name

**A** Warren Hastings

**Analysis** Incorrect. Her father was George Austen. I think the model got confused here because there is a sentence that says, "Austen apparently had unfettered access both to her father's library and that of a family friend, Warren Hastings." This is the only mention of Warren Hastings on this page. I assume the model saw 'father's library' and then the name, 'Warren Hastings' and believed them to be related.

==========================================================================================================================================================================

**Q** what date was she baptized

**A** 5 April

**Analysis** Correct. Even though she was born in December, that winter was particularly harsh so "it was not until 5 April that she was baptised at the local church and christened Jane."

==========================================================================================================================================================================

**Q** where was she educated

**A** She was from then home-educated, until she attended boarding school with her sister from early in 1785 at the Reading Abbey Girls' School

**Analysis** Yes, sort of. She was initially sent to Oxford (with her sister) but that fall sent home after getting sick and was then home-educated until 1785 when she was sent to the Reading Abbey Girls' School. She was only here a short while and before December of 1786 was back at home because the family couldn't afford the boarding school. Her education then came from reading, and guided by her father and brothers.

==========================================================================================================================================================================

**Q** what did she almost die from

**A** Addison's disease

**Analysis** Incorrect. The key word was "almost" and the model seems to have missed that, or did not maket he connection. When she was sent home from Oxford (or perhaps from Southampton, as Ann Cawley, her teacher at Oxford, had taken her and her sister) that fall she had typhus, from which she nearly died. The sentence it should have found was, "That autumn both girls were sent home after catching typhus of which Jane nearly died."

==========================================================================================================================================================================

**Q** who was her Irish friend that she  flirted with

**A** Tom Lefroy

**Analysis** Correct

==========================================================================================================================================================================

**Q** what was her second novel

**A** Sanditon

**Analysis** Incorrect. Sanditon was the last novel she had started before she died. Her second novel was *First Impressions*. The sentence the model should have found was, "In 1796, Austen began a second novel, First Impressions (later published as *Pride and Prejudice*)

==========================================================================================================================================================================

**Q** what was pride and prejudice first titled

**A** First Impressions

**Analysis** Correct

==========================================================================================================================================================================

**Q** where did she live in 1809

**A** Chawton village

**Analysis** Correct. She lived in Chawton Cottage in Chawton village.

==========================================================================================================================================================================

**Q** how much did Austen make from Sense and Sensibility

**A** £140

**Analysis** Correct

==========================================================================================================================================================================

**Q** what is listed as her cause of death

**A** Addison's disease

**Analysis** Yes, and... The majority of biographers list her death as Addison's disease, although her final illness was also described as resulting from Hodgkin's lymphoma.

==========================================================================================================================================================================

**Q** in what year, and by who, was the first dissertation on Austen published

**A** 1883, by George Pellew

**Analysis** Correct. George Pellew was a student at Harvard and was the first one to publish a dissertation on her in 1883.

==========================================================================================================================================================================

**Q** where was her work featured in 2013

**A** many literary anthologies

**Analysis** This is such a strange answer. There's a very clear paragraph that states, "In 2013 Austen's works featured on a series of British postage stamps issued by the Royal Mail to mark the bicentenary of the publication of *Pride and Prejudice.*"

==========================================================================================================================================================================

**Q** what bank note is she featured on

**A** £10

**Analysis** Correct. Issued by the Bank of England and introduced in 2017, Jane Austen replaced Charles Darwin on the 10 pound note.

==========================================================================================================================================================================

**Q** who made a statue of her

**A** Martin Jennings

**Analysis** Correct. Martin Jennings unveiled his statued in October of 2025 at *The Close* which is close to Winchester Cathedral.

==========================================================================================================================================================================

Out of 22 questions 13'ish are correct. This is what, nearly 60% correct? It's not awful, but I honestly did expect slightly better results than these.
